# Evaluating the model

## Set up

In [ ]:
import sys
import json

project_path = "/Users/keira/code/mi-mi-mia/smart-stethoscope"
if project_path not in sys.path:
    sys.path.append(project_path)

## Imports and loading data

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from smart_stethoscope.interface.main import preprocessing
from smart_stethoscope.ml_logic.model import train_xgb_model, train_cnn_model, predict_hybrid

In [2]:
X, mel_spec, y, groups = preprocessing()

print("X shape:", X.shape)
print("Mel shape:", mel_spec.shape)
print("Num labels:", len(y))
print("Num patients:", groups.nunique())


Skipped blacklisted patient 142

Skipped blacklisted patient 182

Skipped blacklisted patient 191

Skipped blacklisted patient 191

Skipped blacklisted patient 191
X shape: (3135, 42)
Mel shape: (3135, 128, 141, 1)
Num labels: 3135
Num patients: 114


## Train/Test/Validation Split

In [3]:
# OUTER split: train_val vs test
sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
outer_splits = list(sgkf.split(X, y, groups))

# Use the first outer fold as test set
train_val_idx, test_idx = outer_splits[0]

# Train_val subset
X_train_val = X.iloc[train_val_idx].reset_index(drop=True)
mel_train_val = mel_spec[train_val_idx]
y_train_val = y[train_val_idx]
groups_train_val = groups.iloc[train_val_idx].reset_index(drop=True)

# Test subset
X_test = X.iloc[test_idx].reset_index(drop=True)
mel_test = mel_spec[test_idx]
y_test = y[test_idx]
groups_test = groups.iloc[test_idx].reset_index(drop=True)

# INNER split: train vs val inside train_val only
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
inner_train_idx, val_idx = next(
    gss.split(X_train_val, y_train_val, groups=groups_train_val)
)

# Train subset
X_train = X_train_val.iloc[inner_train_idx].reset_index(drop=True)
mel_train = mel_train_val[inner_train_idx]
y_train = y_train_val[inner_train_idx]
groups_train = groups_train_val.iloc[inner_train_idx].reset_index(drop=True)

# Validation subset
X_val = X_train_val.iloc[val_idx].reset_index(drop=True)
mel_val = mel_train_val[val_idx]
y_val = y_train_val[val_idx]
groups_val = groups_train_val.iloc[val_idx].reset_index(drop=True)

print("Train chunks:", len(X_train))
print("Val chunks:", len(X_val))
print("Test chunks:", len(X_test))

print("Train patients:", groups_train.nunique())
print("Val patients:", groups_val.nunique())
print("Test patients:", groups_test.nunique())

Train chunks: 1742
Val chunks: 269
Test chunks: 1124
Train patients: 60
Val patients: 16
Test patients: 38


## Train model

Using val for EarlyStopping

In [4]:
xgb_model = train_xgb_model(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val
)

cnn_model = train_cnn_model(
    X_train_img=mel_train,
    y_train=y_train,
    X_val_img=mel_val,
    y_val=y_val
)

print("Models trained ✅")

2026-03-25 20:58:06.117903: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-25 20:58:06.118016: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-25 20:58:06.118028: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-03-25 20:58:06.118192: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-25 20:58:06.118204: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-03-25 20:58:07.270376: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Models trained ✅


# Evaluate on test set

One patient at a time

In [12]:
with open("/Users/keira/code/mi-mi-mia/smart-stethoscope/models/class_names.json") as f:
    class_names = json.load(f)

patient_results = []

for patient_id in groups_test.unique():

    # Get only this patient's test chunks
    mask = (groups_test == patient_id).values

    xgb_features = X_test[mask]
    cnn_features = mel_test[mask]
    y_patient = y_test[mask]

    # True class for this patient
    true_idx = int(y_patient[0])

    # Hybrid prediction for this patient
    preds = predict_hybrid(
        xgb_model=xgb_model,
        cnn_model=cnn_model,
        xgb_df=xgb_features,
        cnn_array=cnn_features,
        class_names=class_names
    )

    final_pred = preds["final_prediction"]

    patient_results.append({
        "patient_id": patient_id,
        "true_idx": true_idx,
        "pred_idx": final_pred["predicted_index"],
        "pred_label": final_pred["predicted_label"],
        "confidence": final_pred["confidence"],
        "correct": int(true_idx == final_pred["predicted_index"])
    })

patient_results_df = pd.DataFrame(patient_results)

print("Test patients evaluated:", len(patient_results_df))
patient_results_df.head()

Test patients evaluated: 38


,patient_id,true_idx,pred_idx,pred_label,confidence,correct
0,109,1,1,Pneumonia,0.933648,1
1,116,0,1,Pneumonia,0.786187,0
2,120,1,1,Pneumonia,0.818452,1
3,122,3,2,Healthy,0.500064,0
4,125,2,1,Pneumonia,0.386505,0


# Evaluate test performance

Report proper patient-level test performance

In [13]:
y_true_patient = patient_results_df["true_idx"]
y_pred_patient = patient_results_df["pred_idx"]

print("Patient-level TEST accuracy:", accuracy_score(y_true_patient, y_pred_patient))
print()
print(classification_report(y_true_patient, y_pred_patient))
print()
print("Confusion matrix:")
print(confusion_matrix(y_true_patient, y_pred_patient))

Patient-level TEST accuracy: 0.631578947368421

              precision    recall  f1-score   support

           0       1.00      0.33      0.50         3
           1       0.63      1.00      0.77        17
           2       0.67      0.60      0.63        10
           3       0.00      0.00      0.00         3
           4       0.00      0.00      0.00         5

    accuracy                           0.63        38
   macro avg       0.46      0.39      0.38        38
weighted avg       0.54      0.63      0.55        38


Confusion matrix:
[[ 1  2  0  0  0]
 [ 0 17  0  0  0]
 [ 0  3  6  0  1]
 [ 0  2  1  0  0]
 [ 0  3  2  0  0]]


/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/keira/.pyenv/versions/3.10.6/envs/smart-stethoscope/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parame

In [17]:
with open("/Users/keira/code/mi-mi-mia/smart-stethoscope/models/class_names.json") as f:
    class_names = json.load(f)

patient_results_df["true_label"] = patient_results_df["true_idx"].map(lambda i: class_names[i])

patient_results_df[[
    "patient_id",
    "true_label",
    "pred_label",
    "confidence",
    "correct"
]].sort_values(["correct", "confidence"], ascending=[True, False]).head(20)

,patient_id,true_label,pred_label,confidence,correct
8,135,URTI,Pneumonia,0.850984,0
29,201,COPD,Pneumonia,0.816699,0
1,116,COPD,Pneumonia,0.786187,0
5,126,Healthy,Bronchiectasis,0.728504,0
32,210,Bronchiectasis,Healthy,0.582142,0
12,148,Bronchiectasis,Healthy,0.564977,0
34,217,Healthy,Pneumonia,0.557959,0
17,164,Bronchiectasis,Pneumonia,0.525096,0
6,127,Healthy,Pneumonia,0.517746,0
7,131,Bronchiectasis,Pneumonia,0.511645,0


### Sanity Check: Chunk Accuracy?

In [15]:
xgb_proba_test = xgb_model.predict_proba(X_test)
chunk_preds_test = xgb_proba_test.argmax(axis=1)

print("Chunk-level TEST accuracy:", accuracy_score(y_test, chunk_preds_test))

Chunk-level TEST accuracy: 0.7224199288256228
